In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 0. SETUP AND DATA LOADING
# ==========================================
# Set style for publication (larger text, clean background)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# Load the dataset
file_path = "twi_qa_answers.xlsx" 
try:
    # Try reading as CSV first to avoid openpyxl errors
    # Change sep=';' to sep=',' if your CSV uses commas
    df = pd.read_excel(file_path) 
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Please make sure the file is in the same folder.")

# Ensure columns have easy-to-use names
df = df.rename(columns={'Marks(0-5)': 'Twi_Score', 'Marks (0-5)': 'English_Score'})

print("Data loaded successfully. Starting analysis...\n")

# ==========================================
# PART 1: CLAUDE'S PERFORMANCE IN TWI ALONE
# ==========================================

# Chart 1.1: Twi Score Distribution
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='Twi_Score', data=df, palette='Blues_d')
plt.title("Figure 1: Distribution of Claude's Scores in Twi", fontsize=14, pad=15)
plt.xlabel("Score (0 = Poor, 5 = Excellent)", fontsize=12)
plt.ylabel("Number of Responses", fontsize=12)

# Add percentages on top of the bars
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('1_twi_score_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# Chart 1.2: Twi Performance by Cultural Topic
twi_topic = df.groupby('topics')['Twi_Score'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='Twi_Score', y='topics', data=twi_topic, palette='viridis')
plt.title("Figure 2: Claude's Average Performance in Twi by Cultural Topic", fontsize=14, pad=15)
plt.xlabel("Average Score (0-5)", fontsize=12)
plt.ylabel("Cultural Topic", fontsize=12)
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig('2_twi_performance_by_topic.png', dpi=300, bbox_inches='tight')
plt.close()


# ==========================================
# PART 2: TWI VS ENGLISH COMPARISON
# ==========================================

# Chart 2.1: Comparative Boxplot (Twi vs English)
plt.figure(figsize=(8, 6))
scores_melted = df.melt(value_vars=['Twi_Score', 'English_Score'], 
                        var_name='Language', value_name='Score')
scores_melted['Language'] = scores_melted['Language'].replace({'Twi_Score': 'Twi', 'English_Score': 'English'})

sns.boxplot(x='Language', y='Score', data=scores_melted, palette=['#3498db', '#e74c3c'], width=0.5)
plt.title("Figure 3: Comparison of Response Quality (Twi vs English)", fontsize=14, pad=15)
plt.xlabel("Prompt and Output Language", fontsize=12)
plt.ylabel("Awarded Score (0-5)", fontsize=12)
plt.ylim(-0.5, 5.5)
plt.tight_layout()
plt.savefig('3_comparison_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()

# Chart 2.2: Comparison by Topic (Grouped Barplot)
topic_scores = df.groupby('topics')[['Twi_Score', 'English_Score']].mean().reset_index()
topic_scores_melted = topic_scores.melt(id_vars='topics', var_name='Language', value_name='Average Score')
topic_scores_melted['Language'] = topic_scores_melted['Language'].replace({'Twi_Score': 'Twi', 'English_Score': 'English'})

# Sort by English performance for better readability
topic_order = topic_scores.sort_values('English_Score', ascending=False)['topics']

plt.figure(figsize=(12, 8))
sns.barplot(x='Average Score', y='topics', hue='Language', data=topic_scores_melted, 
            order=topic_order, palette=['#3498db', '#e74c3c'])
plt.title("Figure 4: Claude's Performance by Topic (Twi vs English)", fontsize=14, pad=15)
plt.xlabel("Average Score (0-5)", fontsize=12)
plt.ylabel("Cultural Topic", fontsize=12)
plt.legend(title='Language', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig('4_comparison_by_topic.png', dpi=300, bbox_inches='tight')
plt.close()

# Chart 2.3: The "Gap" Analysis (English Score - Twi Score) -> VERY USEFUL FOR ARTICLES
topic_scores['Gap (English - Twi)'] = topic_scores['English_Score'] - topic_scores['Twi_Score']
topic_scores_gap = topic_scores.sort_values('Gap (English - Twi)', ascending=True)

plt.figure(figsize=(10, 6))
# Colors change depending on whether Twi is better (rare) or English is better
colors = ['#2ecc71' if gap < 0 else '#e74c3c' for gap in topic_scores_gap['Gap (English - Twi)']]
sns.barplot(x='Gap (English - Twi)', y='topics', data=topic_scores_gap, palette=colors)
plt.title("Figure 5: Performance Gap (AI's Language Penalty)", fontsize=14, pad=15)
plt.xlabel("Score Difference (English Score - Twi Score)", fontsize=12)
plt.ylabel("Cultural Topic", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig('5_language_gap_by_topic.png', dpi=300, bbox_inches='tight')
plt.close()


# ==========================================
# PART 3: CREATING TABLES FOR THE ARTICLE
# ==========================================

# 3.1 Global Statistics Table
summary_stats = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Standard Deviation', 'Minimum', 'Maximum'],
    'Twi': [df['Twi_Score'].mean(), df['Twi_Score'].median(), df['Twi_Score'].std(), df['Twi_Score'].min(), df['Twi_Score'].max()],
    'English': [df['English_Score'].mean(), df['English_Score'].median(), df['English_Score'].std(), df['English_Score'].min(), df['English_Score'].max()]
})

# Round to 2 decimal places
summary_stats = summary_stats.round(2)
summary_stats.to_csv("table_1_summary_statistics.csv", index=False)

# 3.2 Detailed Breakdown by Topic Table
topic_scores = topic_scores.round(2)
topic_scores.to_csv("table_2_topic_breakdown.csv", index=False)

print("--- OVERALL STATISTICS ---")
print(summary_stats.to_string(index=False))
print("\n--- PERFORMANCE BY TOPIC ---")
print(topic_scores.to_string(index=False))
print("\nAll high-quality charts (300 DPI) and CSV tables have been successfully generated and saved to your folder!")

Data loaded successfully. Starting analysis...

--- OVERALL STATISTICS ---
            Metric  Twi  English
              Mean 3.04     4.20
            Median 4.00     5.00
Standard Deviation 1.69     1.21
           Minimum 0.00     0.00
           Maximum 5.00     5.00

--- PERFORMANCE BY TOPIC ---
                topics  Twi_Score  English_Score  Gap (English - Twi)
clothing_and_geography       3.48           4.36                 0.87
 festivals_and_customs       2.96           4.36                 1.39
      food_and_cuisine       2.94           4.38                 1.45
  proverbs_and_sayings       2.72           3.61                 0.89

All high-quality charts (300 DPI) and CSV tables have been successfully generated and saved to your folder!


In [15]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/student25/student25/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
